# Monocular Depth Estimation from a Single Image — U-Net + ResNet Encoder

**What we are building:** a network `f(RGB image) -> depth map`, one depth value per pixel.

**Why this is different from the classification you have done before:**

| Classification | Dense prediction (this) |
|---|---|
| Output: 1 vector of C logits | Output: H×W grid of values |
| Downsample aggressively, throw away spatial detail | Must *recover* spatial detail after downsampling |
| Cross-entropy on one label | A per-pixel regression loss over ~50k pixels |
| Errors are discrete | Errors are geometric — edges, surfaces, planes |

The whole architectural problem of dense prediction is: a CNN encoder is good at *what* is in the
image but destroys *where* (a ResNet18 turns 224×224 into 7×7). A U-Net fixes this by adding a
decoder that upsamples back, and **skip connections** that re-inject the high-resolution features
from the encoder so edges come back sharp.

**Extra wrinkle specific to depth:** from a single 2D image, absolute scale is fundamentally
ambiguous. A dollhouse photographed close up and a real room look identical. So we use a
**scale-invariant loss** rather than MSE — explained in detail in section 4.

---

### Notation used throughout
- `B` batch, `C` channels, `H` height, `W` width
- Tensors are `(B, C, H, W)` — PyTorch's NCHW convention
- Depth is in **metres**, roughly 0.5 m – 10 m for NYU indoor scenes

### Legend for the design-choice flags
- 🟦 **Colab-friendly** — chosen to fit a free T4 in reasonable time; the "real" version is noted.
- 🟩 **Standard practice** — this is genuinely how it is done in papers/production.


## 1. Environment setup

Two things to notice here:

1. **`datasets` is pinned to 3.x on purpose.** The `0jl/NYUv2` dataset is a *script-based* dataset
   — it ships a Python loading script that HF runs for you, which is why it needs
   `trust_remote_code=True`. `datasets` 4.0 **removed** script-based loading entirely, so on a fresh
   Colab (which ships `datasets` 4.x) `trust_remote_code=True` raises an error. Pinning to 3.6.0 is
   the fix.
2. After the pin installs, **Colab will ask you to restart the runtime.** Do it, then skip this cell
   and continue from the next one.


In [ ]:
# --- One-time install. After this finishes, Runtime > Restart session, then continue below. ---
!pip -q install "datasets==3.6.0" "fsspec[http]>=2023.1.0" "huggingface_hub>=0.23" 2>&1 | tail -n 3
!pip -q install matplotlib 2>&1 | tail -n 1
print("Installed. If prompted, RESTART THE RUNTIME now, then run the next cell.")


In [ ]:
import os, math, time, random, json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU only")

# Reproducibility. Note this does NOT make things bit-exact on GPU (cuDNN picks
# nondeterministic algorithms for speed) but it fixes the data split and init.
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


### Mount Google Drive for checkpoints

Colab free-tier sessions die — idle timeout, a closed laptop lid, the 12-hour cap. Anything written
to `/content` dies with them. Drive survives. We will write every checkpoint here.

🟩 **Standard practice:** persisting checkpoints outside the compute node is exactly what real
training does (S3/GCS instead of Drive). The principle is identical: *training state must outlive
the machine.*


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = Path('/content/drive/MyDrive/depth_from_image/checkpoints')
VIZ_DIR  = Path('/content/drive/MyDrive/depth_from_image/viz')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)
print("checkpoints ->", CKPT_DIR)


## 1b. Loading NYU Depth V2

**The dataset:** 1,449 RGB + depth pairs of indoor scenes (bedrooms, kitchens, offices), captured
with a Kinect. The depth channel is the Kinect's measurement, *inpainted* to fill the holes the
sensor leaves on shiny/dark/far surfaces. Range is about 0.5–10 m.

**The fsspec timeout issue.** ~2.9 GB arrives over HTTP through `fsspec`/`aiohttp`, whose default
total-request timeout is 300 s. On a slow Colab link a shard exceeds that and you get
`asyncio.TimeoutError` / `ClientPayloadError` partway through. The fix is to pass a longer
`aiohttp.ClientTimeout` through `storage_options`, which `datasets` forwards to the filesystem
layer. We also wrap the call in a retry loop, because HF's CDN occasionally drops a connection and
a resumed download picks up from cache rather than restarting.


In [ ]:
import aiohttp
from datasets import load_dataset

STORAGE_OPTIONS = {
    "client_kwargs": {
        # total=2h for the whole request, connect=2min for the handshake.
        # sock_read guards against a stalled (not dead) socket.
        "timeout": aiohttp.ClientTimeout(total=7200, connect=120, sock_read=600)
    }
}

def load_nyu(max_retries=4):
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            ds = load_dataset(
                "0jl/NYUv2",
                trust_remote_code=True,
                split="train",
                storage_options=STORAGE_OPTIONS,
            )
            return ds
        except Exception as e:
            last_err = e
            wait = 2 ** attempt
            print(f"[attempt {attempt}/{max_retries}] failed: {type(e).__name__}: {e}")
            print(f"retrying in {wait}s (already-downloaded shards are cached, so this resumes)...")
            time.sleep(wait)
    raise RuntimeError(f"Could not load NYUv2 after {max_retries} attempts") from last_err

raw = load_nyu()
print(raw)


### Inspect the raw records before trusting them

A habit worth building: never write a `Dataset` class against a schema you assumed. Print the actual
types and ranges first. Depth in particular is stored inconsistently across sources — sometimes
float metres, sometimes uint16 millimetres, sometimes a PIL image in mode `I;16` or `F`. Getting
this wrong gives you a model that trains "fine" and predicts nonsense.


In [ ]:
ex = raw[0]
print("keys:", list(ex.keys()))
for k, v in ex.items():
    if isinstance(v, Image.Image):
        a = np.array(v)
        print(f"  {k:8s} PIL mode={v.mode} size={v.size} dtype={a.dtype} "
              f"min={a.min()} max={a.max()}")
    elif isinstance(v, (list, np.ndarray)):
        a = np.asarray(v)
        print(f"  {k:8s} array shape={a.shape} dtype={a.dtype} min={a.min()} max={a.max()}")
    else:
        print(f"  {k:8s} {type(v).__name__}: {str(v)[:60]}")


In [ ]:
RGB_KEY   = "image"
DEPTH_KEY = "depth"

def to_depth_metres(d):
    """Normalise whatever the dataset hands us into a float32 HxW array in METRES.

    Handles: PIL images (any mode), lists, numpy arrays; and auto-detects the
    millimetre-vs-metre convention from the value range.
    """
    if isinstance(d, Image.Image):
        d = np.array(d)
    d = np.asarray(d).astype(np.float32)
    if d.ndim == 3:                 # a stray channel dim
        d = d[..., 0]
    # NYU indoor depth never exceeds ~10 m. If the max is much larger, the units
    # must be millimetres (or the raw uint16 Kinect encoding).
    if d.max() > 100.0:
        d = d / 1000.0
    return d

def to_rgb(img):
    if not isinstance(img, Image.Image):
        img = Image.fromarray(np.asarray(img).astype(np.uint8))
    return img.convert("RGB")

# Sanity-check across a handful of records, not just record 0.
mins, maxs = [], []
for i in range(0, len(raw), max(1, len(raw)//12)):
    d = to_depth_metres(raw[i][DEPTH_KEY])
    mins.append(float(d[d > 0].min())); maxs.append(float(d.max()))
print(f"depth min over samples: {min(mins):.3f} m   max: {max(maxs):.3f} m")
print(f"RGB size: {to_rgb(raw[0][RGB_KEY]).size}")

MIN_DEPTH, MAX_DEPTH = 0.5, 10.0   # the standard NYU evaluation range


### Look at the data

Depth maps are single-channel, so they are rendered with a colormap. `magma` is a good default:
perceptually uniform, and dark→bright reads intuitively as near→far (we invert it so **bright = near**,
which matches how most depth papers present results).

Notice while looking: the floor forms a smooth gradient from near (bottom) to far, walls are flat
planes, and object boundaries are sharp depth *discontinuities*. Those discontinuities are the hard
part — they are exactly what the skip connections exist to preserve.


In [ ]:
def show_pair(rgb, depth, ax_rgb, ax_d, title=""):
    ax_rgb.imshow(rgb); ax_rgb.set_title(title or "RGB"); ax_rgb.axis("off")
    valid = depth[depth > 0]
    im = ax_d.imshow(depth, cmap="magma_r",
                     vmin=np.percentile(valid, 1), vmax=np.percentile(valid, 99))
    ax_d.set_title(f"depth  [{valid.min():.1f}–{depth.max():.1f} m]"); ax_d.axis("off")
    return im

n = 4
idxs = random.sample(range(len(raw)), n)
fig, axes = plt.subplots(2, n, figsize=(4*n, 7))
for col, i in enumerate(idxs):
    r = to_rgb(raw[i][RGB_KEY]); d = to_depth_metres(raw[i][DEPTH_KEY])
    show_pair(np.array(r), d, axes[0, col], axes[1, col], title=f"sample {i}")
plt.suptitle("NYU Depth V2 — RGB (top) vs ground-truth depth (bottom, bright = near)", y=0.98)
plt.tight_layout(); plt.show()


---
## 2. Dataset, transforms and DataLoader

Three decisions to make here, and each one has a depth-specific trap.

### 2a. Resize — and the interpolation trap
Native NYU is 480×640. We train at **192×256** (same 3:4 aspect).

🟦 **Colab-friendly:** 192×256 with ResNet18 lets a batch of 16 fit comfortably in a T4's 15 GB and
keeps an epoch under a minute. Real depth papers train at 416×544 or full resolution, often with
batch 8–16 across multiple GPUs. Lower resolution costs you thin structures (chair legs, lamp cords)
— the model literally cannot represent them.

**The trap:** resize RGB with **bilinear**, but resize depth with **nearest-neighbour**.
Bilinear averaging across a depth discontinuity invents a pixel that is the mean of "0.8 m" (the
chair) and "4 m" (the wall behind) = 2.4 m — a surface that exists nowhere in the scene. Do that
everywhere along every object boundary and you are training the model to hallucinate ramps at every
edge. Nearest-neighbour just picks one of the two real values.

### 2b. Normalization
- **RGB:** the ImageNet mean/std, because our encoder's pretrained weights were learned on inputs
  standardised that way. Feed it raw [0,1] pixels and the pretrained features are being evaluated
  off-distribution. 🟩 Standard practice.
- **Depth:** we do *not* normalize it. It stays in metres and the loss handles scale in log space
  (section 4). We only build a **validity mask**: Kinect depth has invalid (zero) pixels, and the
  standard NYU protocol only evaluates 0.5–10 m. Those pixels must be excluded from the loss, or the
  model spends its capacity learning to predict sensor failures.

### 2c. Augmentation — and why most of them are dangerous here
With only ~1,000 training images, augmentation matters. But a depth map is not a label you can
transform freely; it encodes **3D geometry tied to the camera**. The rule: an augmentation is safe
only if there exists a real camera pose / real scene that would have produced the augmented pair.

| Augmentation | Safe? | Why |
|---|---|---|
| **Horizontal flip** (flip both) | ✅ | Equivalent to a mirrored room. Geometry stays consistent, and indoor scenes have no strong left/right prior. |
| **Colour jitter / brightness** (RGB only) | ✅ | Photometric only — changing the lightbulb doesn't move the walls. Depth map untouched. |
| **Rotation** | ⚠️ **risky** | Two separate problems. (1) *Interpolation*: any angle that isn't a multiple of 90° resamples the depth map and blends across discontinuities — the same phantom-surface problem as bilinear resize, now everywhere. (2) *Prior destruction*: depth has a very strong vertical structure — floor near at the bottom, ceiling above, gravity-aligned walls. The network exploits "pixels lower in the frame are usually closer." Rotate and you teach it a relationship that no real camera produces, since people hold cameras roughly level. Small rotations (±5°) are used in some papers, but they are a genuine trade-off, not free. |
| **Vertical flip** | ❌ | Puts the floor on the ceiling. No real camera does this. Destroys the vertical prior outright. |
| **Random crop** | ~ | Fine for *relative* depth, but cropping changes the effective field of view, and FOV is exactly the cue that relates apparent size to distance. For metric depth it's a mild distribution shift. |
| **Random resize / scale** | ❌ (naively) | Zooming in makes objects look closer, but the stored depth values are unchanged — you've created an image-depth pair that is physically impossible. Correct scale augmentation requires rescaling the depth values too. |

So: **horizontal flip + colour jitter.** That is genuinely enough here.


In [ ]:
IMG_H, IMG_W = 192, 256
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


class NYUDepthDataset(Dataset):
    '''Wraps the HF dataset; returns (rgb, depth, mask) as float32 tensors.

    rgb   : (3, H, W)  ImageNet-normalised
    depth : (1, H, W)  metres, raw (NOT normalised)
    mask  : (1, H, W)  True where depth is a valid measurement to supervise on
    '''

    def __init__(self, hf_dataset, indices, train=False):
        self.ds = hf_dataset
        self.indices = list(indices)
        self.train = train
        self.jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                             saturation=0.2, hue=0.02)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        rec = self.ds[self.indices[i]]
        rgb = to_rgb(rec[RGB_KEY])
        depth = to_depth_metres(rec[DEPTH_KEY])

        # --- resize: bilinear for RGB, NEAREST for depth (see 2a) ---
        rgb = rgb.resize((IMG_W, IMG_H), Image.BILINEAR)
        depth = np.array(
            Image.fromarray(depth).resize((IMG_W, IMG_H), Image.NEAREST)
        )

        rgb = torch.from_numpy(np.array(rgb)).permute(2, 0, 1).float() / 255.0
        depth = torch.from_numpy(depth).unsqueeze(0).float()

        # --- augmentation (train split only) ---
        if self.train:
            if random.random() < 0.5:
                # Flip BOTH, or the pair stops corresponding. A classic silent bug:
                # flip only the image and the model still trains, just badly.
                rgb = torch.flip(rgb, dims=[2])
                depth = torch.flip(depth, dims=[2])
            if random.random() < 0.5:
                rgb = self.jitter(rgb)          # photometric only — depth untouched

        rgb = (rgb - IMAGENET_MEAN) / IMAGENET_STD

        # --- validity mask ---
        mask = (depth > MIN_DEPTH) & (depth < MAX_DEPTH)
        # Clamp so that log() downstream can never see a zero or a negative.
        depth = depth.clamp(min=MIN_DEPTH, max=MAX_DEPTH)
        return rgb, depth, mask


### Splitting 1,449 images into train / val / test

There is no official split for the labeled subset, so we make one. Two points:

- **Shuffle with a fixed seed.** The records are ordered by scene, so a naive first-80% split would
  put entire scene *types* only in train and others only in test — you'd be measuring the wrong thing.
- **Three splits, not two.** Val is what you look at while tuning (and therefore what you slowly
  overfit to, through your own decisions). Test gets touched **once**, at the end. 🟩 Standard practice.

A caveat worth knowing: even a random split leaks a little, because NYU's 1,449 frames come from
464 scenes, so near-duplicate frames of the same room can land on both sides. A stricter protocol
splits *by scene*. 🟦 We use the simple random split for clarity; just don't read the test numbers as
if they were true generalisation to unseen rooms.


In [ ]:
N = len(raw)
perm = np.random.RandomState(SEED).permutation(N)
n_train, n_val = int(0.80 * N), int(0.10 * N)
train_idx = perm[:n_train]
val_idx   = perm[n_train:n_train + n_val]
test_idx  = perm[n_train + n_val:]
print(f"total {N}  ->  train {len(train_idx)} | val {len(val_idx)} | test {len(test_idx)}")

BATCH_SIZE  = 16      # fits a T4 at 192x256 with ResNet18 + AMP
NUM_WORKERS = 2       # Colab gives 2 vCPUs; more workers just thrash

train_ds = NYUDepthDataset(raw, train_idx, train=True)
val_ds   = NYUDepthDataset(raw, val_idx,   train=False)
test_ds  = NYUDepthDataset(raw, test_idx,  train=False)

common = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True,  **common)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False, **common)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False, **common)

rgb_b, depth_b, mask_b = next(iter(train_loader))
print("rgb  ", tuple(rgb_b.shape),   rgb_b.dtype,   f"[{rgb_b.min():.2f}, {rgb_b.max():.2f}]")
print("depth", tuple(depth_b.shape), depth_b.dtype, f"[{depth_b.min():.2f}, {depth_b.max():.2f}] m")
print("mask ", tuple(mask_b.shape),  f"{mask_b.float().mean():.1%} of pixels valid")


### Sanity-check a real batch, after augmentation

Always look at what actually comes out of the loader, not at what you think comes out. This catches
the flip-mismatch bug, the wrong-units bug, and the mask-is-all-False bug in about ten seconds.
We un-normalize the RGB to display it.


In [ ]:
def denorm(rgb_t):
    return (rgb_t.cpu() * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

k = 4
fig, axes = plt.subplots(3, k, figsize=(4*k, 9.5))
for j in range(k):
    axes[0, j].imshow(denorm(rgb_b[j])); axes[0, j].set_title("RGB (augmented)"); axes[0, j].axis("off")
    d = depth_b[j, 0].numpy()
    axes[1, j].imshow(d, cmap="magma_r", vmin=MIN_DEPTH, vmax=d.max()); axes[1, j].set_title("depth (m)"); axes[1, j].axis("off")
    axes[2, j].imshow(mask_b[j, 0].numpy(), cmap="gray"); axes[2, j].set_title("valid mask"); axes[2, j].axis("off")
plt.tight_layout(); plt.show()


---
## 3. The model: ResNet encoder + U-Net decoder

### The shape story
A ResNet18 applied to a 192×256 input produces a pyramid of feature maps, each half the resolution
and (mostly) double the channels of the one before:

```
input           3 x 192 x 256
 |
 +-- x0  (conv1+bn+relu)        64 x  96 x 128     /2   <-- skip
 +-- x1  (maxpool + layer1)     64 x  48 x  64     /4   <-- skip
 +-- x2  (layer2)              128 x  24 x  32     /8   <-- skip
 +-- x3  (layer3)              256 x  12 x  16     /16  <-- skip
 +-- x4  (layer4)              512 x   6 x   8     /32  <-- bottleneck
```

`x4` knows *what* the scene is but has thrown away almost all *where*: 6×8 cells for the whole
image. The decoder walks back up, and at each step concatenates the matching encoder map:

```
x4 (512, /32)
  up -> cat x3 (256) -> conv -> 256 (/16)
  up -> cat x2 (128) -> conv -> 128 (/8)
  up -> cat x1  (64) -> conv ->  64 (/4)
  up -> cat x0  (64) -> conv ->  32 (/2)
  up -> (no skip)    -> conv ->  16 (/1)
  1x1 conv -> 1 x 192 x 256
```

**Why the skips matter, concretely:** the bottleneck can tell you "there is a chair roughly here."
Only `x0` still knows *which pixel* the chair's edge falls on. Without skips you get a blurry,
plausible-looking depth map with mush at every boundary. This is the single most important idea in
U-Net and it's worth deleting the skips once, training 3 epochs, and seeing the difference yourself.

### Upsampling: `interpolate` + conv, not `ConvTranspose2d`
Transposed convolutions produce checkerboard artifacts when kernel size isn't divisible by stride.
Nearest-neighbour upsample followed by a regular 3×3 conv avoids that entirely, and is what most
modern dense-prediction decoders do. 🟩 Standard practice.

### 🔑 Freeze the encoder, or fine-tune from the start?

**Answer: fine-tune, but with a short frozen warm-up and a lower learning rate on the encoder.**
Here is the reasoning, because it generalises to every transfer-learning problem:

1. **Why not freeze permanently?** ImageNet features were trained for *semantic* discrimination —
   "is this fur or fabric?" Depth needs *geometric* cues: vanishing lines, surface slant, relative
   size, occlusion boundaries. Those overlap with ImageNet features but are not the same thing. A
   frozen encoder caps your accuracy; every good depth model fine-tunes the backbone.
2. **Why not fine-tune from step 0 at full LR?** At initialisation the decoder is random, so the
   first gradients flowing back into the encoder are essentially noise, scaled up by a large loss.
   A few hundred steps of that can wreck pretrained features you can't recover — "catastrophic
   forgetting." With 1,000 images you cannot afford to re-learn them.
3. **The practical recipe** (what we do):
   - Epochs 0–1: encoder **frozen**, decoder trains alone until it stops emitting noise.
   - Epoch 2 onward: **unfreeze**, encoder LR = 1e-4, decoder LR = 1e-3 (a 10× *discriminative*
     learning rate). Early layers already encode edges/textures that are universal; the later
     blocks need more adaptation, and the decoder needs the most.

🟦 **Colab-friendly:** ResNet18 (11M params) over ResNet34 (21M). ResNet34 is a drop-in change on
the line below and gives modestly better results for ~1.7× the time. Real systems today use much
heavier backbones (ConvNeXt, Swin, DINOv2-ViT), which is precisely what makes them expensive.

**Also 🟦:** we keep encoder BatchNorm layers in `eval()` mode while frozen. BN statistics update
even when weights are frozen, which quietly shifts the pretrained features under you — a subtle bug
worth knowing about.


In [ ]:
class ConvBlock(nn.Module):
    '''3x3 conv -> BN -> ReLU, twice. The standard U-Net building block.'''
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    '''Upsample x2, concatenate the encoder skip, then convolve.'''
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv = ConvBlock(in_ch + skip_ch, out_ch)

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        if skip is not None:
            # Odd input sizes make the upsampled map differ from the skip by 1px.
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="nearest")
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class DepthUNet(nn.Module):
    def __init__(self, backbone="resnet18", pretrained=True,
                 min_depth=MIN_DEPTH, max_depth=MAX_DEPTH):
        super().__init__()
        self.min_depth, self.max_depth = min_depth, max_depth

        if backbone == "resnet18":
            weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
            net = torchvision.models.resnet18(weights=weights)
            chs = [64, 64, 128, 256, 512]
        elif backbone == "resnet34":
            weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
            net = torchvision.models.resnet34(weights=weights)
            chs = [64, 64, 128, 256, 512]
        else:
            raise ValueError(backbone)

        # Split the ResNet into the five stages we want to tap.
        self.stem    = nn.Sequential(net.conv1, net.bn1, net.relu)   # /2   64
        self.pool    = net.maxpool
        self.layer1, self.layer2 = net.layer1, net.layer2            # /4, /8
        self.layer3, self.layer4 = net.layer3, net.layer4            # /16, /32

        self.up4 = UpBlock(chs[4], chs[3], 256)   # /32 -> /16
        self.up3 = UpBlock(256,    chs[2], 128)   # /16 -> /8
        self.up2 = UpBlock(128,    chs[1],  64)   # /8  -> /4
        self.up1 = UpBlock(64,     chs[0],  32)   # /4  -> /2
        self.up0 = UpBlock(32,          0,  16)   # /2  -> /1, no skip left
        self.head = nn.Conv2d(16, 1, 1)

    # --- encoder freezing -------------------------------------------------
    def set_encoder_trainable(self, flag: bool):
        for m in (self.stem, self.layer1, self.layer2, self.layer3, self.layer4):
            for p in m.parameters():
                p.requires_grad = flag
        self._encoder_frozen = not flag

    def train(self, mode=True):
        super().train(mode)
        # Keep frozen encoder BN layers in eval mode so running stats don't drift.
        if mode and getattr(self, "_encoder_frozen", False):
            for m in (self.stem, self.layer1, self.layer2, self.layer3, self.layer4):
                m.eval()
        return self

    def encoder_parameters(self):
        for m in (self.stem, self.layer1, self.layer2, self.layer3, self.layer4):
            yield from m.parameters()

    def decoder_parameters(self):
        for m in (self.up4, self.up3, self.up2, self.up1, self.up0, self.head):
            yield from m.parameters()

    # --- forward ----------------------------------------------------------
    def forward(self, x):
        x0 = self.stem(x)               # 64,  /2
        x1 = self.layer1(self.pool(x0)) # 64,  /4
        x2 = self.layer2(x1)            # 128, /8
        x3 = self.layer3(x2)            # 256, /16
        x4 = self.layer4(x3)            # 512, /32

        d = self.up4(x4, x3)
        d = self.up3(d,  x2)
        d = self.up2(d,  x1)
        d = self.up1(d,  x0)
        d = self.up0(d)
        raw = self.head(d)              # (B,1,H,W) unbounded

        # Map the unbounded output into [min_depth, max_depth] *in log space*.
        # Why log space: the loss (section 4) works on log-depth, and a log-uniform
        # parameterisation gives the network equal resolution per *ratio* of depth
        # rather than per metre -- 0.5->1.0 m matters as much as 5->10 m.
        # sigmoid also guarantees we never emit a non-positive depth, so log() is safe.
        log_min, log_max = math.log(self.min_depth), math.log(self.max_depth)
        return torch.exp(log_min + torch.sigmoid(raw) * (log_max - log_min))


model = DepthUNet(backbone="resnet18", pretrained=True).to(DEVICE)
n_all = sum(p.numel() for p in model.parameters())
n_enc = sum(p.numel() for p in model.encoder_parameters())
print(f"total params {n_all/1e6:.1f}M  (encoder {n_enc/1e6:.1f}M, decoder {(n_all-n_enc)/1e6:.1f}M)")

with torch.no_grad():
    out = model(rgb_b[:2].to(DEVICE))
print("input ", tuple(rgb_b[:2].shape), "-> output", tuple(out.shape),
      f"range [{out.min():.2f}, {out.max():.2f}] m")
assert out.shape[-2:] == rgb_b.shape[-2:], "output resolution must match input"
print("shapes OK")


---
## 4. The loss function: scale-invariant log loss (SILog)

### Why plain MSE is the wrong tool

**Problem 1 — scale is genuinely ambiguous from one image.** A photo of a real kitchen and a photo of
a perfect 1:10 scale model of that kitchen are *identical images*. Nothing in the pixels determines
absolute size. A model trained with MSE is punished hard for a prediction that is perfectly correct
up to a single global multiplier — i.e. for getting the entire 3D structure of the scene right. It
therefore hedges toward the dataset's mean depth, and you get flat, washed-out predictions.

**Problem 2 — MSE weights metres, but we care about ratios.** Being 0.5 m wrong on a mug 0.6 m away
is a catastrophic error (you'd grab air). Being 0.5 m wrong on a wall 9 m away is excellent. MSE
scores them identically, so gradients are dominated by far pixels, which are also the ones with the
noisiest Kinect ground truth. Working in **log space** fixes this: `log(a) - log(b) = log(a/b)`, so
an error in log space *is* a relative error.

### The loss, mathematically

Let `d_i = log(pred_i) - log(gt_i)` over the `n` valid pixels. Then (Eigen et al., NeurIPS 2014):

$$
\mathcal{L} \;=\; \frac{1}{n}\sum_i d_i^2 \;-\; \frac{\lambda}{n^2}\Big(\sum_i d_i\Big)^2
\;=\; \operatorname{mean}(d^2) - \lambda\,\operatorname{mean}(d)^2
$$

Read the two terms:

- `mean(d²)` is just **MSE in log space** — the average squared *relative* error. This solves
  problem 2.
- `mean(d)²` is the squared **mean log error**. If your whole prediction is off by one constant
  factor `s` (i.e. `pred = s · gt` everywhere), then every `d_i = log s`, so `mean(d) = log s`.
  Subtracting this term **removes exactly that global-scale component from the penalty**.

And there is a neat identity that makes what's happening obvious. With `λ = 1`:

$$
\operatorname{mean}(d^2) - \operatorname{mean}(d)^2 \;=\; \operatorname{Var}(d)
$$

So at `λ = 1` the loss is literally **the variance of the log-error** — it does not care about the
mean of the error at all, only about whether the errors are *consistent* with each other. That is
what "scale-invariant" means: get the relative structure right, and the constant offset is free.

**Why λ = 0.85 rather than 1.0?** At λ=1 you make scale entirely free, and the model never learns
even roughly how big rooms are — useful for a pure-shape model, but then your predictions are
unitless. λ=0.85 keeps 15% of the mean-error penalty, so the model is nudged toward the right
absolute scale while still being mostly forgiven for scale mistakes. 🟩 λ=0.85 with the
`α·sqrt(...)` wrapper is the standard formulation used by Eigen, BTS, AdaBins and DepthFormer.

**The `α·sqrt(·)` wrapper** (α=10): taking the square root makes the loss behave like a *standard
deviation* rather than a variance — the gradient doesn't collapse as the loss shrinks — and α=10
just puts the number in a range where default learning rates work. Cosmetic, but universal.

### Optional second term: gradient matching
SILog alone is happy with a smooth, slightly blurry map. Adding a small penalty on the difference of
**spatial gradients** between prediction and ground truth explicitly rewards sharp, correctly-placed
edges. Five lines, visibly sharper results. 🟩 Standard in modern depth work (MiDaS uses a
multi-scale version of exactly this).


In [ ]:
class SILogLoss(nn.Module):
    '''Scale-invariant log loss (Eigen et al. 2014), optionally + gradient matching.

    loss = alpha * sqrt( mean(d^2) - lam * mean(d)^2 ),   d = log(pred) - log(gt)
    '''
    def __init__(self, lam=0.85, alpha=10.0, grad_weight=0.5):
        super().__init__()
        self.lam, self.alpha, self.grad_weight = lam, alpha, grad_weight

    def forward(self, pred, target, mask):
        # Only supervise on valid pixels. Masking by indexing (not multiplying by 0)
        # keeps the *means* correct -- dividing by H*W instead of n would silently
        # scale the loss down by whatever fraction of pixels are invalid.
        p = pred[mask].clamp(min=1e-3)
        t = target[mask].clamp(min=1e-3)
        d = torch.log(p) - torch.log(t)

        var_term = (d ** 2).mean() - self.lam * (d.mean() ** 2)
        # Numerical guard: var_term is >= 0 mathematically, but fp16 rounding can
        # make it a tiny negative number, and sqrt of that is NaN.
        loss = self.alpha * torch.sqrt(var_term.clamp(min=1e-8))

        if self.grad_weight > 0:
            loss = loss + self.grad_weight * self._gradient_loss(pred, target, mask)
        return loss

    @staticmethod
    def _gradient_loss(pred, target, mask):
        '''L1 on the difference of first-order spatial differences, in log space.

        Penalises the prediction having an edge where the GT does not (and vice
        versa), which is what makes boundaries crisp instead of ramped.
        '''
        lp = torch.log(pred.clamp(min=1e-3))
        lt = torch.log(target.clamp(min=1e-3))
        diff = lp - lt
        m = mask.float()

        # Gradient is only meaningful where BOTH neighbouring pixels are valid.
        gx, gx_m = diff[..., :, 1:] - diff[..., :, :-1], m[..., :, 1:] * m[..., :, :-1]
        gy, gy_m = diff[..., 1:, :] - diff[..., :-1, :], m[..., 1:, :] * m[..., :-1, :]
        denom = gx_m.sum() + gy_m.sum() + 1e-6
        return ((gx.abs() * gx_m).sum() + (gy.abs() * gy_m).sum()) / denom


criterion = SILogLoss(lam=0.85, alpha=10.0, grad_weight=0.5)

# --- Demonstrate the scale-invariance property ------------------------------
gt = depth_b[:2].to(DEVICE)
m  = mask_b[:2].to(DEVICE)
mse = nn.MSELoss()
for s in [1.0, 1.5, 3.0]:
    pred = gt * s   # perfect relative structure, wrong global scale by factor s
    print(f"pred = {s:>3.1f} x gt   |  SILog {criterion(pred, gt, m).item():7.4f}"
          f"   |  plain MSE {mse(pred[m], gt[m]).item():8.4f}")
print("\nSILog barely moves (only the 15% scale penalty). MSE explodes -- "
      "it punishes a geometrically perfect prediction.")


### Metrics: what each number actually tells you

We report the four standard NYU metrics. They are *not* redundant — each fails differently:

| Metric | Formula | Reads as | Sensitive to |
|---|---|---|---|
| **AbsRel** | `mean(\|pred − gt\| / gt)` | "average error as a fraction of true depth" — 0.15 means typically 15% off | relative accuracy everywhere; **near** pixels count as much as far ones |
| **RMSE** | `sqrt(mean((pred − gt)²))` | "typical error in metres" | **far** pixels and outliers — a few badly-wrong background pixels dominate it |
| **RMSE log** | `sqrt(mean((log pred − log gt)²))` | RMSE in relative terms | like AbsRel but punishes big errors harder |
| **δ₁** | `% of pixels with max(p/g, g/p) < 1.25` | "fraction of pixels within 25% of correct" — **higher is better** | overall hit rate; robust to a few disasters |

Read AbsRel and RMSE **together**. Low AbsRel + high RMSE = good on nearby objects, bad on far walls.
High AbsRel + low RMSE = the reverse. For reference: a strong 2024 model gets AbsRel ≈ 0.09 and
δ₁ ≈ 0.92 on NYU. This notebook's model should land around **AbsRel 0.16–0.20, δ₁ 0.72–0.80**. 🟦
That gap is resolution, backbone size and training length — not a different idea.

**Median scaling.** Because we trained with a mostly scale-invariant loss, the prediction's global
scale is only loosely pinned. The standard evaluation protocol for such models multiplies the
prediction by `median(gt)/median(pred)` per image before scoring, which measures *structure* rather
than *absolute calibration*. We report both, so you can see how much of the error is pure scale.


In [ ]:
@torch.no_grad()
def compute_metrics(pred, gt, mask, median_scale=True):
    '''Returns a dict of the standard NYU depth metrics for one batch.'''
    p, g = pred[mask], gt[mask]
    if p.numel() == 0:
        return {}
    if median_scale:
        p = p * (torch.median(g) / torch.median(p))
    p = p.clamp(MIN_DEPTH, MAX_DEPTH)

    thresh = torch.max(p / g, g / p)
    return {
        "abs_rel":  ((p - g).abs() / g).mean().item(),
        "rmse":     torch.sqrt(((p - g) ** 2).mean()).item(),
        "rmse_log": torch.sqrt(((torch.log(p) - torch.log(g)) ** 2).mean()).item(),
        "d1":       (thresh < 1.25).float().mean().item(),
        "d2":       (thresh < 1.25 ** 2).float().mean().item(),
        "d3":       (thresh < 1.25 ** 3).float().mean().item(),
    }


@torch.no_grad()
def evaluate(model, loader, median_scale=True, max_batches=None):
    model.eval()
    acc, n = {}, 0
    for bi, (rgb, gt, msk) in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break
        rgb, gt, msk = rgb.to(DEVICE), gt.to(DEVICE), msk.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            pred = model(rgb)
        met = compute_metrics(pred.float(), gt, msk, median_scale)
        if not met:
            continue
        for k, v in met.items():
            acc[k] = acc.get(k, 0.0) + v
        n += 1
    return {k: v / max(n, 1) for k, v in acc.items()}


def fmt_metrics(m):
    return (f"AbsRel {m['abs_rel']:.4f} | RMSE {m['rmse']:.4f} m | "
            f"RMSElog {m['rmse_log']:.4f} | d1 {m['d1']:.4f} | d2 {m['d2']:.4f}")


---
## 5. Training

### Checkpointing that actually survives a disconnect
Saving `model.state_dict()` alone is not enough to *resume* — you also need the optimizer state
(Adam's momentum buffers), the scheduler position, the AMP scaler state, and the epoch number.
Restarting from weights only means Adam re-warms its moment estimates from zero and the first few
hundred steps after a resume are noticeably worse. We save all of it, every epoch, to Drive, plus a
separate `best.pt` keyed on validation AbsRel. 🟩 Standard practice.

### Mixed precision (AMP)
A T4 has fp16 tensor cores. `torch.autocast` runs the convolutions in fp16 while keeping the
loss/reductions in fp32, roughly halving memory and nearly doubling throughput. `GradScaler` exists
because fp16 gradients underflow to zero: it multiplies the loss by a large factor before backward,
then divides it out of the gradients. Note the T4 is Turing, so it has fp16 but **not** bf16 —
that's why we specify `dtype=torch.float16` explicitly.

### Schedule
🟦 20 epochs of ~72 steps each. At 192×256 with ResNet18 + AMP this is roughly **8–15 minutes total**
on a T4. Real training runs 50+ epochs at higher resolution on much more data (NYU's *full* raw
set is 120k frames, not 1.4k). With 1,159 training images the model will start overfitting; watching
val loss diverge from train loss is itself part of the lesson.

Cosine annealing with a short linear warm-up, and the two-phase encoder freezing described in §3.


In [ ]:
EPOCHS        = 20
FREEZE_EPOCHS = 2          # encoder frozen for this many epochs at the start
LR_DECODER    = 1e-3
LR_ENCODER    = 1e-4       # 10x lower -- discriminative learning rates
WEIGHT_DECAY  = 1e-4

model = DepthUNet(backbone="resnet18", pretrained=True).to(DEVICE)

# Two param groups so the encoder and decoder can have different learning rates.
optimizer = torch.optim.AdamW([
    {"params": list(model.encoder_parameters()), "lr": LR_ENCODER},
    {"params": list(model.decoder_parameters()), "lr": LR_DECODER},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = len(train_loader)
total_steps     = EPOCHS * steps_per_epoch
warmup_steps    = steps_per_epoch  # one epoch of linear warm-up

def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * prog))       # cosine down to 0

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
criterion = SILogLoss(lam=0.85, alpha=10.0, grad_weight=0.5)

print(f"{steps_per_epoch} steps/epoch x {EPOCHS} epochs = {total_steps} steps")


In [ ]:
LAST_CKPT = CKPT_DIR / "last.pt"
BEST_CKPT = CKPT_DIR / "best.pt"

def save_checkpoint(path, epoch, best_metric, history):
    # Write to a temp file then rename: if Colab dies mid-write, the previous
    # checkpoint is still intact rather than half-overwritten.
    tmp = Path(str(path) + ".tmp")
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "best_metric": best_metric,
        "history": history,
        "config": {"img_h": IMG_H, "img_w": IMG_W, "backbone": "resnet18",
                   "min_depth": MIN_DEPTH, "max_depth": MAX_DEPTH},
    }, tmp)
    tmp.replace(path)

def load_checkpoint(path):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    scaler.load_state_dict(ck["scaler"])
    print(f"resumed from {path} @ epoch {ck['epoch']} (best AbsRel {ck['best_metric']:.4f})")
    return ck["epoch"] + 1, ck["best_metric"], ck.get("history", [])

start_epoch, best_metric, history = 0, float("inf"), []
if LAST_CKPT.exists():
    start_epoch, best_metric, history = load_checkpoint(LAST_CKPT)
else:
    print("no checkpoint found -- starting fresh")


### The periodic visualization

Numbers tell you *whether* it improved; pictures tell you *how*. What to watch for, epoch by epoch:

1. **Epochs 1–2:** an almost-constant map, maybe a vertical gradient. The model has learned the
   dataset's mean depth and the floor prior. This is normal, not a bug.
2. **Epochs 3–8:** large regions separate — walls go far, foreground furniture goes near. Edges are
   still soft.
3. **Epochs 9+:** boundaries sharpen (that's the gradient-matching term), and thin structures start
   appearing.

If it's still uniform at epoch 5, something is wrong — check the mask isn't all-False and that the
learning rate isn't 0. Each figure is also written to Drive so you can flip through them later.


In [ ]:
@torch.no_grad()
def visualize(model, dataset, epoch, n=4, save=True, tag="val"):
    model.eval()
    picks = list(range(0, len(dataset), max(1, len(dataset) // n)))[:n]
    fig, axes = plt.subplots(n, 3, figsize=(11, 3.1 * n))
    if n == 1:
        axes = axes[None, :]
    for r, idx in enumerate(picks):
        rgb, gt, msk = dataset[idx]
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            pred = model(rgb[None].to(DEVICE)).float()[0, 0].cpu()

        g = gt[0].numpy()
        # Share a colour scale between pred and GT, or you are comparing pictures
        # that have been independently rescaled and everything looks great.
        mnp = msk[0].numpy()
        vmin, vmax = float(g[mnp].min()), float(g[mnp].max())
        axes[r, 0].imshow(denorm(rgb));                axes[r, 0].set_title("input RGB")
        axes[r, 1].imshow(pred.numpy(), cmap="magma_r", vmin=vmin, vmax=vmax)
        axes[r, 1].set_title("predicted depth")
        axes[r, 2].imshow(g, cmap="magma_r", vmin=vmin, vmax=vmax)
        axes[r, 2].set_title("ground truth")
        for c in range(3):
            axes[r, c].axis("off")
    plt.suptitle(f"epoch {epoch}  ({tag})  -- bright = near", y=1.0)
    plt.tight_layout()
    if save:
        fig.savefig(VIZ_DIR / f"{tag}_epoch_{epoch:03d}.png", dpi=80, bbox_inches="tight")
    plt.show()


In [ ]:
def train_one_epoch(epoch):
    # Phase 1: encoder frozen so random decoder gradients can't damage pretrained features.
    # Phase 2: unfreeze and fine-tune at the lower encoder LR.
    if epoch < FREEZE_EPOCHS:
        model.set_encoder_trainable(False)
        phase = "frozen encoder"
    else:
        model.set_encoder_trainable(True)
        phase = "fine-tuning"
    model.train()

    running, t0 = 0.0, time.time()
    for i, (rgb, gt, msk) in enumerate(train_loader):
        rgb, gt, msk = (rgb.to(DEVICE, non_blocking=True),
                        gt.to(DEVICE, non_blocking=True),
                        msk.to(DEVICE, non_blocking=True))

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            pred = model(rgb)
        # Compute the loss in fp32: log() and the variance subtraction are exactly
        # the operations where fp16 precision bites.
        loss = criterion(pred.float(), gt, msk)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                      # unscale before clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running += loss.item()
        if i % 20 == 0:
            lrs = [g["lr"] for g in optimizer.param_groups]
            print(f"  ep{epoch:02d} [{i:3d}/{steps_per_epoch}] loss {loss.item():.4f} "
                  f"| lr enc {lrs[0]:.2e} dec {lrs[1]:.2e} | {phase}")
    return running / steps_per_epoch, time.time() - t0


for epoch in range(start_epoch, EPOCHS):
    train_loss, secs = train_one_epoch(epoch)
    val = evaluate(model, val_loader, median_scale=True)
    history.append({"epoch": epoch, "train_loss": train_loss, **val})

    print(f"\nepoch {epoch:02d} | train loss {train_loss:.4f} | {fmt_metrics(val)} | {secs:.0f}s")

    save_checkpoint(LAST_CKPT, epoch, best_metric, history)
    if val["abs_rel"] < best_metric:
        best_metric = val["abs_rel"]
        save_checkpoint(BEST_CKPT, epoch, best_metric, history)
        print(f"  -> new best AbsRel {best_metric:.4f}, saved to {BEST_CKPT.name}")

    if epoch % 2 == 0 or epoch == EPOCHS - 1:
        visualize(model, val_ds, epoch, n=3)
    print("-" * 90)

print(f"\ndone. best val AbsRel = {best_metric:.4f}")


### Training curves

Two things to read off these plots:
- **Train loss still falling while val AbsRel flattens or rises** → overfitting. With 1,159 images
  this is expected around epoch 12–15. The `best.pt` checkpoint protects you from it.
- **δ₁ rising steeply then plateauing** → the model has captured the coarse layout and is now only
  refining. Further gains need more capacity or more data, not more epochs.


In [ ]:
hist = history
ep = [h["epoch"] for h in hist]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(ep, [h["train_loss"] for h in hist], marker="o"); ax[0].set_title("train loss (SILog + grad)")
ax[1].plot(ep, [h["abs_rel"] for h in hist], marker="o", color="tab:red")
ax[1].plot(ep, [h["rmse"] for h in hist],    marker="s", color="tab:orange", label="RMSE (m)")
ax[1].set_title("val AbsRel (red) / RMSE"); ax[1].legend(["AbsRel", "RMSE (m)"])
ax[2].plot(ep, [h["d1"] for h in hist], marker="o", color="tab:green")
ax[2].set_title(r"val $\delta_1$  (higher is better)")
for a in ax:
    a.set_xlabel("epoch"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### Final evaluation on the held-out test split

This is the one time we touch `test_idx`. We load `best.pt` (selected on *val*) and report both
median-scaled and unscaled numbers — the gap between them is precisely the absolute-scale error that
the scale-invariant loss deliberately did not force the model to fix.


In [ ]:
if BEST_CKPT.exists():
    ck = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"])
    print(f"loaded best.pt from epoch {ck['epoch']}")

test_scaled   = evaluate(model, test_loader, median_scale=True)
test_unscaled = evaluate(model, test_loader, median_scale=False)
print("\nTEST (median-scaled)  :", fmt_metrics(test_scaled))
print("TEST (raw, no scaling):", fmt_metrics(test_unscaled))
print("\nGap between the two = how far off the model's absolute scale calibration is.")

visualize(model, test_ds, epoch=999, n=4, tag="test")


---
## 6. Inference on your own photos

Run the cell below and upload anything — indoor shots, but **especially outdoor/street photos**,
since that is where you will learn the most.

**What to expect, and why.** The model has seen 1,159 images, all indoor, all from a Kinect, all
within 0.5–10 m, all from roughly chest height. On a street photo it will:

- still produce something depth-*shaped*, because the low-level cues it learned (occlusion, texture
  gradient, vanishing lines, "lower in frame = nearer") are genuinely general;
- **compress everything into 0–10 m** — it structurally cannot output more, because our sigmoid
  parameterisation bounds it there. A building 80 m away and a car 15 m away both saturate to "10 m,"
  so the far field looks flat. This is a hard architectural limit, not a training failure;
- confuse sky. Sky is at infinity and never appears in NYU; the model has no concept for it and will
  usually guess "wall."

That contrast is the point of this section: distinguish *the model didn't learn this cue* from
*the model cannot express this output*.


In [ ]:
from google.colab import files
import io

@torch.no_grad()
def predict_image(pil_img, model):
    '''Full inference path for one arbitrary image. Note it mirrors the training
    transforms exactly -- any mismatch here is the #1 cause of "it worked in
    training but is garbage at inference".'''
    model.eval()
    img = pil_img.convert("RGB")
    orig_w, orig_h = img.size

    x = img.resize((IMG_W, IMG_H), Image.BILINEAR)
    x = torch.from_numpy(np.array(x)).permute(2, 0, 1).float() / 255.0
    x = ((x - IMAGENET_MEAN) / IMAGENET_STD)[None].to(DEVICE)

    with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        pred = model(x).float()
    # Upsample the prediction back to the original resolution for display.
    pred = F.interpolate(pred, size=(orig_h, orig_w), mode="bilinear", align_corners=False)
    return pred[0, 0].cpu().numpy()


def show_prediction(pil_img, depth, title=""):
    fig, ax = plt.subplots(1, 3, figsize=(16, 5))
    ax[0].imshow(pil_img); ax[0].set_title("input"); ax[0].axis("off")
    im = ax[1].imshow(depth, cmap="magma_r"); ax[1].set_title("predicted depth (bright = near)")
    ax[1].axis("off"); plt.colorbar(im, ax=ax[1], fraction=0.046, label="metres")
    # Overlay makes it obvious whether object boundaries actually line up.
    ax[2].imshow(pil_img); ax[2].imshow(depth, cmap="magma_r", alpha=0.55)
    ax[2].set_title("overlay"); ax[2].axis("off")
    plt.suptitle(f"{title}   predicted range: {depth.min():.2f}-{depth.max():.2f} m")
    plt.tight_layout(); plt.show()


uploaded = files.upload()          # pick one or more images
for name, data in uploaded.items():
    img = Image.open(io.BytesIO(data))
    d = predict_image(img, model)
    show_prediction(img, d, title=name)
    frac_far = float((d > 9.0).mean())
    print(f"{name}: {frac_far:.1%} of pixels saturated near the 10 m ceiling "
          f"{'<-- out-of-range scene, the model is clipping' if frac_far > 0.25 else ''}")


### Optional: a quick diagnostic on your own photo

A useful trick for judging an out-of-distribution prediction: look at the **histogram** of predicted
depths. A healthy indoor prediction spreads across the range with a couple of modes (floor, walls).
A saturated outdoor prediction piles up against 10 m. Same model, and the histogram tells you
immediately which failure you are looking at.


In [ ]:
for name, data in uploaded.items():
    img = Image.open(io.BytesIO(data))
    d = predict_image(img, model)
    plt.figure(figsize=(6, 3))
    plt.hist(d.ravel(), bins=60, range=(MIN_DEPTH, MAX_DEPTH))
    plt.axvline(MAX_DEPTH, color="r", ls="--", label="architectural ceiling")
    plt.title(f"predicted depth histogram -- {name}"); plt.xlabel("metres"); plt.legend()
    plt.tight_layout(); plt.show()


---
## 7. Summary: what this model does well, where it breaks, and where to go next

### What it does well
- **Coarse scene layout.** Floor/wall/ceiling separation, which surfaces recede, which objects are
  in front — reliably correct on indoor scenes. This is the bulk of the δ₁ score.
- **Relative ordering.** "The chair is nearer than the bookshelf" is almost always right, which is
  exactly what the scale-invariant loss optimised for.
- **Large object boundaries**, thanks to the skip connections plus the gradient-matching term.
  Delete the skips and re-train for three epochs to see how much of this is them.
- **Transfer from ImageNet.** A pretrained encoder on 1,159 images reaches usable results in ~10
  minutes. Train the same architecture from scratch (`pretrained=False`) and it will be visibly worse
  — a clean experiment worth running once.

### Where it struggles
- **Absolute scale.** Deliberately: λ=0.85 forgives 85% of the global-scale error. Compare the
  median-scaled and raw test numbers to quantify it.
- **Thin structures** — chair legs, lamp stands, cables. At 192×256 a chair leg is 1–2 pixels wide
  and survives neither the nearest-neighbour depth resize nor a /32 bottleneck. 🟦 Resolution.
- **Reflective and transparent surfaces** (mirrors, windows, TV screens). Partly the model, partly
  the *data*: the Kinect fails on these too, so the "ground truth" is inpainted guesswork and the
  model is being trained on a hallucination.
- **Out-of-distribution scenes.** Outdoor/street photos get relative structure that is often
  surprisingly reasonable, but hard-clipped at 10 m by the sigmoid parameterisation, and sky is
  interpreted as a wall.
- **Overfitting.** 1,159 images is tiny. Expect val metrics to stall around epoch 12–15 while train
  loss keeps dropping.

### Three concrete next steps (described, not implemented)

**1. Train on KITTI for outdoor/street scenes.**
KITTI is driving footage with LiDAR ground truth out to ~80 m. The work is more than a dataset
swap: (a) raise `MAX_DEPTH` to 80 and keep the log-space parameterisation, which already handles a
160× range gracefully; (b) KITTI's LiDAR is *sparse* — only ~5% of pixels have a value, so the
validity mask becomes essential rather than a detail, and the gradient-matching term needs rethinking
because neighbouring valid pixels are rare; (c) crop to the standard Eigen crop, since the top of
each frame is sky with no LiDAR returns at all. A model trained on both NYU and KITTI with a shared
backbone is the natural follow-on, and immediately raises the question of how to mix two datasets
with incompatible depth scales — which is exactly what MiDaS's scale-and-shift-invariant loss was
invented to solve.

**2. Swap the ResNet encoder for a ViT / DINOv2 backbone.**
The current biggest weakness is global context: a ResNet18's receptive field at the bottleneck still
doesn't cleanly span the image, so the model reasons somewhat locally. A ViT attends globally from
layer one, and self-supervised DINOv2 features are known to encode geometry remarkably well — this
is precisely why Depth Anything v1/v2 are built on them. Concretely: keep the U-Net decoder, replace
the encoder with `dinov2_vits14`, tap four intermediate transformer blocks as the "pyramid" (they are
all at /14 resolution, so you reshape and resample them to the four scales the decoder expects — the
DPT trick). 🟦 Expect ~3–4× the training time and a fixed patch-multiple input size; a ViT-S/14 is
still feasible on a T4.

**3. Add a second, geometry-aware loss and evaluate in 3D.**
Depth metrics are a proxy — what you actually care about is the reconstructed surface. Two additions
worth making: (a) a **surface-normal loss**, computing normals from the predicted depth via its
spatial gradients and penalising the angle against GT normals — this directly targets flatness of
walls and floors, which pixel-wise losses barely constrain; (b) back-project the predicted depth to a
**point cloud** using NYU's camera intrinsics (fx≈518.86, fy≈519.47, cx≈325.58, cy≈253.74) and look
at it in 3D. Predictions that score well on AbsRel can still look like a warped bowl in 3D, and you
will not see it any other way.

### One more experiment, if you want to internalise the core idea
Run three short trainings with everything else identical:
1. plain MSE on raw depth, 2. plain MSE on log-depth, 3. the SILog loss.
Compare the val AbsRel *and* look at the predictions. You will see MSE-on-depth produce flat,
mean-seeking maps; log-MSE fix most of it; and SILog add the last bit of structure. That single
experiment is the whole argument of section 4, made concrete.
